# 03 — Training protocol and safety

**Tujuan.** Audit konfigurasi sebelum GPU training, tanpa menjalankannya. Paper §IV-B hlm. 91509 menyatakan split 8:1:1, input 512, LR 0,000186, batch 16, 80 epoch. Seed 42, AdamW, CE, AMP, clipping, aturan checkpoint, dan strategi oversampling adalah `ASSUMED`. Batch efektif 16 dengan microbatch lebih kecil mengubah statistik BatchNorm.

## Alur yang benar-benar dijalankan

`load_protocol` dan preflight memeriksa konfigurasi, manifest, hash source, serta split sebelum campaign. `dataset_for` membuat `FundusSplit` hanya untuk split diminta; `__getitem__` membaca gambar, menjalankan `preprocess_fundus`, augmentasi train bila aktif, lalu mengubah array menjadi tensor CHW float `[0,1]`. `split_loader` memakai seed `42+epoch`, sampler `balanced_train_indices` hanya pada train balanced, dan tidak mengulang validation/test. Loss adalah `CrossEntropyLoss`; optimizer `AdamW` dengan weight decay 0, LR 0,000186. Checkpoint terbaik dipilih dari **validation macro-F1**, dengan validation loss sebagai pemecah seri; final test adalah evaluasi terpisah pada checkpoint terpilih.

| Campaign aktual | Epoch | LR | Microbatch × akumulasi = efektif | Presisi | Balancing train | Seleksi checkpoint |
| --- | ---: | ---: | --- | --- | --- | --- |
| APTOS balanced | 80 | 0.000186 | 16 × 1 = 16 | AMP FP16 | train_oversample_uniform | validation_macro_f1 |
| DDR recovery | 80 | 0.000186 | 8 × 2 = 16 | FP32 | train_oversample_uniform | validation_macro_f1 |

Sumber konfigurasi: [APTOS config](../../results/APTOS2019/full_oversampling_amp_fp16_80epochs/config.json) dan [DDR config](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/config.json). Field `campaign: balanced_v1` di DDR `config.json` menamai keluarga protokol; identitas run final test yang dicatat dalam ringkasan adalah `cadvit_tables_1_5_balanced_fp32_math_mb8_diag/ddr_full`. [FORK_PROVENANCE](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/FORK_PROVENANCE.json) menghubungkan fase pemulihan dengan run asal epoch 42. Angka paper 80 epoch, batch 16, LR 0,000186, dan split 8:1:1 ialah **EXPLICIT IN PAPER**; seed, AdamW, loss, clip 1, akumulasi, dan pemilihan macro-F1 merupakan **IMPLEMENTATION ASSUMPTIONS**. Catatan V1 menggunakan `WeightedRandomSampler`; V2 BG10 tidak memakai sampler itu. Tiga generasi tidak disamakan sebagai satu protokol.

In [1]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "reproduction/cadvit/model.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ["CAD_VIT_ARTIFACT_ROOT"]).expanduser().resolve() if os.environ.get("CAD_VIT_ARTIFACT_ROOT") else None
print("Artifact root:", ARTIFACT_ROOT if ARTIFACT_ROOT else "PENDING (set CAD_VIT_ARTIFACT_ROOT)")

Artifact root: PENDING (set CAD_VIT_ARTIFACT_ROOT)


In [2]:
from reproduction.cadvit.experiments import Protocol, CAMPAIGNS, load_protocol, matrix
print("Paper explicit: input=512, LR=0.000186, batch=16, epochs=80, split=8:1:1")
print("Runner defaults (assumptions marked in docs):", Protocol())
if ARTIFACT_ROOT:
    for campaign in CAMPAIGNS:
        try:
            protocol, fingerprint = load_protocol(ARTIFACT_ROOT, campaign)
            print(campaign, "AVAILABLE", fingerprint, "balancing", protocol.balancing)
        except (FileNotFoundError, ValueError) as exc:
            print(campaign, "PENDING:", exc)
else:
    print("PENDING: no local campaign artifact root; no training is launched")

Paper explicit: input=512, LR=0.000186, batch=16, epochs=80, split=8:1:1
Runner defaults (assumptions marked in docs): Protocol(seed=42, epochs=80, learning_rate=0.000186, effective_batch_size=16, micro_batch_size=16, accumulation=1, preprocessing='paper_described_historical_v1', augment_train=True, balancing='none', optimizer='AdamW', weight_decay=0.0, loss='CrossEntropyLoss', amp=True, grad_clip_norm=1.0, checkpoint_metric='validation_macro_f1', checkpoint_tiebreak='validation_loss', num_workers=0)
PENDING: no local campaign artifact root; no training is launched


## Kurva training yang tersimpan

**APTOS balanced — ACTUAL EXPERIMENTAL RESULT.** [Kurva 80 epoch](../report_figures/aptos_balanced_80_epoch_curves.svg) berasal dari [`history.jsonl`](../../results/APTOS2019/full_oversampling_amp_fp16_80epochs/history.jsonl). Best validation macro-F1 **0,661421356** pada epoch **24**; final-test macro-F1 dilaporkan terpisah di notebook 04.

![APTOS balanced: training loss dan metrik validation sepanjang 80 epoch](../report_figures/aptos_balanced_80_epoch_curves.svg)

**DDR recovery — ACTUAL EXPERIMENTAL RESULT.** Panel berikut membaca [`history.jsonl`](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/history.jsonl), masing-masing 80 titik. **Batas protokol berada setelah epoch 42**: [FORK_PROVENANCE](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/FORK_PROVENANCE.json) mencatat fork ke FP32 Math SDPA, microbatch 8 dan akumulasi 2 pada epoch 43. Best validation macro-F1 **0,653069359** pada epoch **69**. Garis sambung melintasi batas ini adalah urutan rekaman, bukan seri protokol homogen.

![DDR recovery train loss, batas protokol setelah epoch 42](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/train_loss.png)
![DDR recovery validation loss, batas protokol setelah epoch 42](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/validation_loss.png)
![DDR recovery validation accuracy, batas protokol setelah epoch 42](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/validation_accuracy.png)
![DDR recovery validation macro-F1, best epoch 69](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/validation_macro_f1.png)

Grafik berikut memakai 80 nilai asli dan menandai batas protokol secara eksplisit. Panel PNG asli di atas tidak memiliki garis vertikal; keduanya tidak mengubah log training.



PNG asli tidak menggambar garis vertikal perubahan protokol; caption ini menetapkan batas yang harus dipakai saat membacanya. Riwayat DDR V2 80 epoch lain melibatkan FP64 clipping dan checkpoint epoch 72; kurvanya ada pada [arsip figur historis](../report_figures/historical_v2_ddr_training_curves.png), bukan bagian recovery terbaru. NaN pada upaya DDR yang lebih awal dan percobaan AMP/FP32 dicatat sebagai riwayat diagnostik di [Eksperimen 3](../experiments/03_experiment_balanced_training_and_ddr_recovery.ipynb); penyebab tunggal tidak dibuktikan. Akumulasi mempertahankan ukuran batch efektif nominal 16, tetapi statistik dan jalur numeriknya tidak identik dengan microbatch 16. Perbedaan validation dan final test tidak digunakan untuk memilih ulang checkpoint.

**Catatan output lama.** Pesan `PENDING` pada sel protokol di atas berarti `CAD_VIT_ARTIFACT_ROOT` tidak disetel saat sel itu dieksekusi; ia bukan status training APTOS/DDR yang ringkasan dan kurvanya telah dikomit.

**Metode dan batasan.** `paper_v1` di KC sudah dibuat dan tidak boleh ditimpa. `balanced_v1` harus memiliki direktori, protokol, fingerprint, dan checkpoint terpisah. Runner melakukan preflight manifest tanpa membuka gambar test, memilih checkpoint dengan validation macro-F1, dan hanya mengakses test lewat perintah final-test eksplisit. Pilot satu epoch memakai `--max-epochs-per-invocation 1`; perintah lengkap ada di rencana eksperimen.